> **Added in 2026 / 2026'da eklendi:** This is my notebook exactly as I ran it on Kaggle for the Garanti BBVA Data Day case study (May 2024, finished 11th of 51). Outputs are cleared because they printed raw competition rows. `src/original.py` replays it line for line, and [`01_what_went_wrong.ipynb`](01_what_went_wrong.ipynb) walks through its mistakes.
>
> Mayıs 2024'te Garanti BBVA Data Day için Kaggle'da çalıştırdığım notebook, olduğu gibi (51 kişi içinde 11.). Ham yarışma satırlarını yazdırdığı için çıktılarını temizledim. `src/original.py` onu satır satır tekrar ediyor; hatalarını [`01_what_went_wrong.ipynb`](01_what_went_wrong.ipynb) anlatıyor.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df_train = pd.read_csv("/kaggle/input/garanti-bbva-data-day-case-study/train.csv")
df_train

# EDA

In [ ]:
df_train['CUSTNBR'].nunique()

In [ ]:
# Find duplicate customers (those appearing more than once)
duplicate_customers = df_train['CUSTNBR'].value_counts()
duplicate_customers = duplicate_customers[duplicate_customers > 1]

# Print the list of duplicate customer numbers (optional)
print(duplicate_customers)
print(len(duplicate_customers))

# Sample a few rows for a specific customer from the list
# Replace '12345' with a customer number from your duplicate_customers index
sample_rows = df_train[df_train['CUSTNBR'] == 145485.0]  # Sample 5 rows

# Display the sampled rows
print(sample_rows)

In [ ]:
df_train.columns

In [ ]:
df_train.info()

In [ ]:
df_train.describe()

In [ ]:
df_train['TRAN_DATE'] = pd.to_datetime(df_train['TRAN_DATE'])
df_train['month'] = df_train['TRAN_DATE'].dt.month
#df_train['day'] = df_train['TRAN_DATE'].dt.day
df_train.drop(columns=['TRAN_DATE'], inplace=True)
df_train.drop(columns=['CONTPAIDAMNT07'], inplace=True)

***I dropped 'TRAN_DATE' column and I took just month info from it and I added new 'month' column  ***

In [ ]:
df_train.isnull().sum()

In [ ]:
#df_train.fillna(df_train.mean(), inplace=True)
df_train.fillna(0, inplace=True)

***I filled 0 to NaN values***

In [ ]:
corr_matrix = df_train.corr()
import seaborn as sns
import matplotlib.pyplot as plt

# Create a heatmap
plt.figure(figsize=(20, 16))  # You can adjust the size as needed
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', cbar=True, square=True)
plt.title('Correlation Matrix of df_train')
plt.show()

In [ ]:

filtered_df = df_train[df_train['ADDCONTAMNT'] > 0]


max_month_per_id = filtered_df.groupby('CUSTNBR')['month'].max()


filtered_df = filtered_df[filtered_df.apply(lambda row: row['month'] == max_month_per_id[row['CUSTNBR']], axis=1)]


result_df = filtered_df[['CUSTNBR', 'ADDCONTAMNT']]


print(result_df)

**I wanted to create a dataframe with customers with 'ADDCONTAMNT' positive, some customers had multiple transactions. Therefore, if the customer made more than one transaction, I added the most recent transaction to the dataframe thanks to 'month' column***.**I did this because I will replace these users' predictions with their last transactions. Because I caught a correlation during EDA.**

# ***Trying Feature Combinations --> I choosed some features and add in feature_names list then I create a algorithm works that choose at least 3 feature and fit model print rmse shortly. I wanted to see best feature combinations***

In [ ]:
feature_names = feature_names = ['MAXMATEVR', 'CONTDEBTNUM','CONTPAIDAMNT00','RTRNDESVAMNT','CONTCOLLAMNT','month','CONTPAIDAMNT01','CONTPAIDAMNT02','CONTPAIDAMNT03','CONTPAIDAMNT04','DEBTAVG00', 'ADDCONTAMNT']
new_df = df_train[feature_names].copy()

train_df_m = new_df[new_df['month'] < 11]
test_df_m = new_df[new_df['month'] == 11]

y_train = train_df_m['ADDCONTAMNT']
X_train_m = train_df_m.drop(columns=['ADDCONTAMNT'])
y_test = test_df_m['ADDCONTAMNT']
X_test_m = test_df_m.drop(columns=['ADDCONTAMNT'])

X_train=X_train_m.drop(columns=['month'])
X_test=X_test_m.drop(columns=['month'])                      


In [ ]:
''''''
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import itertools
import numpy as np

# Assuming X_train, X_test, y_train, and y_test are your datasets

# Define a list of feature names or indices
feature_names = list(X_train.columns)  # Assuming X_train is a DataFrame

# Initialize a list to store the results
results = []

# Loop over different number of features to select (starting from 3)
for k in range(3, len(feature_names) + 1):
    # Generate all combinations of k features
    feature_combinations = itertools.combinations(feature_names, k)
    
    # Loop over each combination of features
    for features in feature_combinations:
        # Select the columns corresponding to the current combination of features
        X_train_subset = X_train[list(features)]
        X_test_subset = X_test[list(features)]
        
        # Train a model using the current combination of features
        model = LinearRegression()
        model.fit(X_train_subset, y_train)
        
        # Make predictions on the test set
        y_pred = model.predict(X_test_subset)
        y_pred_clipped = np.where(y_pred < 0, 0, y_pred)
        # Calculate RMSE
        rmse = np.sqrt(mean_squared_error(y_test, y_pred_clipped))
        
        # Record the results (combination of features and corresponding RMSE)
        results.append((features, rmse))

# Sort the results by RMSE
results.sort(key=lambda x: x[1])

# Print all attempts and their scores
for features, rmse in results:
    print("Features:", features)
    print("RMSE:", rmse)
    print("-----------------------------")

In [ ]:
y = df_train['ADDCONTAMNT']
X = df_train.drop(columns=['CUSTNBR','ADDCONTAMNT'])

In [ ]:
from sklearn.preprocessing import StandardScaler

# Instantiate the StandardScaler
scaler = StandardScaler()

# Fit the scaler to the training data and transform the training data
X_train_scaled = scaler.fit_transform(X)


In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import math

# Assuming X and y are your features and target variable
# Choose the number of folds for cross-validation
k = 10
kf = KFold(n_splits=k, shuffle=True, random_state=42)

# Initialize an empty list to store the evaluation metric (e.g., Mean Squared Error) for each fold
mse_scores = []

# Instantiate the Linear Regression model
model = LinearRegression()

# Perform k-fold cross-validation
for train_index, test_index in kf.split(X_train_scaled):
    # Split the data into training and testing sets for this fold
    X_train, X_test = X_train_scaled[train_index], X_train_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Fit the model on the training data
    model.fit(X_train, y_train)
    
    # Make predictions on the test data
    y_pred = model.predict(X_test)
    
    # Calculate the evaluation metric (e.g., Mean Squared Error) for this fold
    mse = mean_squared_error(y_test, y_pred)
    mse_scores.append(math.sqrt(mse))

# Calculate the average Mean Squared Error across all folds
avg_mse = np.mean(mse_scores)
print("Average Mean Squared Error:", avg_mse)

In [ ]:
mse_scores

In [ ]:
df_test = pd.read_csv("/kaggle/input/garanti-bbva-data-day-case-study/test_input.csv")
df_test

In [ ]:
test_in_train = df_test['CUSTNBR'].isin(df_train['CUSTNBR'])

# Display the result
print(sum(test_in_train))

In [ ]:
df_test['TRAN_DATE'] = pd.to_datetime(df_test['TRAN_DATE'],format='%d.%m.%Y')
df_test['month'] = df_test['TRAN_DATE'].dt.month
#df_test['day'] = df_test['TRAN_DATE'].dt.day
df_test.drop(columns=['TRAN_DATE'], inplace=True)

In [ ]:
df_test.fillna(0, inplace=True)

In [ ]:
submission_df = pd.DataFrame()
submission_df['Id'] = df_test["CUSTNBR"]

In [ ]:
df_test = df_test.drop(columns=["CUSTNBR"])

In [ ]:
df_test = scaler.fit_transform(df_test)

In [ ]:

pred = model.predict(df_test)
pred

In [ ]:
submission_df['Predicted'] = pred
submission_df

In [ ]:
submission_df['Predicted'] = submission_df['Predicted'].apply(lambda x: max(0, x))
submission_df

***Negative predictions converted to 0***

In [ ]:
import pandas as pd

# Assuming df1 is the first dataset with 'CUSTNMBR' and 'ADDCONTAMNT' columns
# and df2 is the second dataset with 'Id' and 'Prediction' columns

# Merge the datasets on the 'CUSTNMBR' column with a left join
merged_df = pd.merge(submission_df, result_df, left_on='Id', right_on='CUSTNBR', how='left')

# Update 'Prediction' only where 'CUSTNMBR' matches 'Id'
merged_df.loc[merged_df['CUSTNBR'] == merged_df['Id'], 'Predicted'] = merged_df['ADDCONTAMNT']

# Drop the 'CUSTNMBR' and 'ADDCONTAMNT' columns as they are no longer needed
merged_df.drop(['CUSTNBR', 'ADDCONTAMNT'], axis=1, inplace=True)

# Print the resulting dataset
print(merged_df)

***After the Predict I want to change some customers prediction with last transactions***

In [ ]:
merged_df.to_csv('/kaggle/working/submission.csv', index=False)